In [ ]:
import duckdb
import pandas as pd
import logging
import os
from pathlib import Path

# Setup logging
logging.basicConfig(
    filename='../logs/pipeline.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [ ]:
DB_PATH = '../data/conflict_prediction.duckdb' 
# check for file structure
os.makedirs('../data', exist_ok=True)
os.makedirs('../logs', exist_ok=True)

# setting up logger and connection
con = duckdb.connect(DB_PATH)
logger.info("Connected to DuckDB database")
print("Connected to DuckDB database")

DATA_DIR = '../data/raw' 

# files
FILES = {
    'acled': f'{DATA_DIR}Middle-East_aggregated_data_up_to-2026-03-07.csv',
    'gdelt': f'{DATA_DIR}gdelt_middle_east.csv',
    'sipri': f'{DATA_DIR}SIPRI_current_USD.csv',
    'sipri_per_capita': f'{DATA_DIR}SIPRI_per_capita.csv',
    'fsi_2019': f'{DATA_DIR}fsi-2019.csv',
    'fsi_2020': f'{DATA_DIR}fsi-2020.csv',
    'fsi_2021': f'{DATA_DIR}fsi-2021.csv',
    'fsi_2022': f'{DATA_DIR}fsi-2022.csv',
    'fsi_2023': f'{DATA_DIR}fsi-2023.csv',
    'acled_index': f'{DATA_DIR}ACLED_ci.csv',
    'world_bank': f'{DATA_DIR}World_bank_hist.csv',
}
# making it easier to work with data files

# Verify files exist
for name, path in FILES.items():
    exists = os.path.exists(path)
    status = "FOUND" if exists else "MISSING"
    print(f"  {status}: {name} -> {path}")
    if not exists:
        logger.warning(f"File not found: {path}")

In [ ]:
# Middle East countries with code mappings -- this will make all of the joins super easy
countries_data = {
    'country_code': ['BHR','IRN','IRQ','ISR','JOR','KWT','LBN','OMN','PSE','QAT','SAU','SYR','TUR','ARE','YEM'],
    'country_name': ['Bahrain','Iran','Iraq','Israel','Jordan','Kuwait','Lebanon','Oman','Palestine','Qatar','Saudi Arabia','Syria','Turkey','United Arab Emirates','Yemen'],
    'fips_code': ['BA','IR','IZ','IS','JO','KU','LE','MU','WE','QA','SA','SY','TU','AE','YM'],
    'acled_name': ['Bahrain','Iran','Iraq','Israel','Jordan','Kuwait','Lebanon','Oman','Palestine','Qatar','Saudi Arabia','Syria','Turkey','United Arab Emirates','Yemen'],
    'region': ['Middle East'] * 15,
}

countries_df = pd.DataFrame(countries_data)

# Add World Bank classification if file exists
try:
    wb = pd.read_csv(FILES['world_bank'])
    # Map to our countries
    for idx, row in countries_df.iterrows():
        match = wb[wb['Country'].str.contains(row['country_name'], case=False, na=False)]
        if len(match) > 0:
            countries_df.loc[idx, 'income_group'] = match.iloc[0].get('FY26', 'Unknown')
            countries_df.loc[idx, 'lending_category'] = 'Unknown'
    logger.info("Added World Bank classifications")
except Exception as e:
    # error handling
    logger.warning(f"Could not load World Bank data: {e}")
    countries_df['income_group'] = 'Unknown'
    countries_df['lending_category'] = 'Unknown'

con.execute("DROP TABLE IF EXISTS countries")
con.execute("""
    CREATE TABLE countries AS SELECT * FROM countries_df
""")
print(f"COUNTRIES table created: {con.execute('SELECT COUNT(*) FROM countries').fetchone()[0]} rows")
logger.info("COUNTRIES table created")

In [ ]:
try:
    acled = pd.read_csv(FILES['acled'])
    
    # Rename columns to clean format
    acled.columns = [c.lower().replace(' ', '_') for c in acled.columns]
    
    # Rename week to event_date if needed
    if 'week' in acled.columns:
        acled = acled.rename(columns={'week': 'event_date'})
    
    # Add country_code by mapping from COUNTRIES
    country_map = dict(zip(countries_df['acled_name'], countries_df['country_code']))
    acled['country_code'] = acled['country'].map(country_map)
    
    con.execute("DROP TABLE IF EXISTS acled_events")
    con.execute("""
        CREATE TABLE acled_events AS SELECT * FROM acled
    """)
    
    row_count = con.execute('SELECT COUNT(*) FROM acled_events').fetchone()[0]
    print(f"ACLED_EVENTS table created: {row_count:,} rows")
    logger.info(f"ACLED_EVENTS table created: {row_count} rows")
    
except Exception as e:
    print(f"ERROR loading ACLED: {e}")
    logger.error(f"ACLED loading failed: {e}")

In [ ]:
try:
    gdelt = pd.read_csv(FILES['gdelt'], dtype=str, low_memory=False)
    
    # Add country_code by mapping FIPS -> ISO
    fips_map = dict(zip(countries_df['fips_code'], countries_df['country_code']))
    gdelt['country_code'] = gdelt['ActionGeo_CountryCode'].map(fips_map)
    
    # Convert numeric columns
    for col in ['GoldsteinScale', 'NumMentions', 'NumSources', 'NumArticles', 
                'AvgTone', 'ActionGeo_Lat', 'ActionGeo_Long']:
        gdelt[col] = pd.to_numeric(gdelt[col], errors='coerce')

    # Filter only countries we care about
    gdelt_me = gdelt.dropna(subset=['country_code'])
    
    con.execute("DROP TABLE IF EXISTS gdelt_events")
    con.execute("""
        CREATE TABLE gdelt_events AS SELECT * FROM gdelt_me
    """)
    
    row_count = con.execute('SELECT COUNT(*) FROM gdelt_events').fetchone()[0]
    print(f"GDELT_EVENTS table created: {row_count:,} rows")
    logger.info(f"GDELT_EVENTS table created: {row_count} rows")
    
except Exception as e:
    print(f"ERROR loading GDELT: {e}")
    logger.error(f"GDELT loading failed: {e}")

In [ ]:
try:
    sipri = pd.read_csv(FILES['sipri'])
    
    # SIPRI data is wide format (one column per year) — melt to long format
    id_cols = ['Country', 'Notes']
    year_cols = [c for c in sipri.columns if c not in id_cols]
    
    sipri_long = sipri.melt(
        id_vars=['Country'], 
        value_vars=year_cols,
        var_name='year', 
        value_name='spending_current_usd'
    )
    
    # Clean up
    sipri_long['year'] = pd.to_numeric(sipri_long['year'], errors='coerce')
    sipri_long['spending_current_usd'] = pd.to_numeric(
        sipri_long['spending_current_usd'].replace(['...', 'xxx', '..'], pd.NA), 
        errors='coerce'
    )
    sipri_long = sipri_long.dropna(subset=['year', 'spending_current_usd'])
    sipri_long['year'] = sipri_long['year'].astype(int)
    
    # Filter to Middle East countries and map codes
    me_names = countries_df['country_name'].tolist()
    # SIPRI may use slightly different names, so do fuzzy matching
    name_map = {}
    for sipri_name in sipri_long['Country'].unique():
        for me_name in me_names:
            if me_name.lower() in sipri_name.lower() or sipri_name.lower() in me_name.lower():
                name_map[sipri_name] = me_name
                break
    
    sipri_long['matched_name'] = sipri_long['Country'].map(name_map)
    sipri_me = sipri_long.dropna(subset=['matched_name']).copy()
    
    country_code_map = dict(zip(countries_df['country_name'], countries_df['country_code']))
    sipri_me['country_code'] = sipri_me['matched_name'].map(country_code_map)
    sipri_me = sipri_me[['country_code', 'year', 'spending_current_usd']]
    
    con.execute("DROP TABLE IF EXISTS military_spending")
    con.execute("""
        CREATE TABLE military_spending AS SELECT * FROM sipri_me
    """)
    
    row_count = con.execute('SELECT COUNT(*) FROM military_spending').fetchone()[0]
    print(f"MILITARY_SPENDING table created: {row_count:,} rows")
    logger.info(f"MILITARY_SPENDING table created: {row_count} rows")
    
except Exception as e:
    print(f"ERROR loading SIPRI: {e}")
    logger.error(f"SIPRI loading failed: {e}")

In [ ]:
try:
    fsi_frames = []
    fsi_keys = [k for k in FILES if k.startswith('fsi_')]
    
    for key in fsi_keys:
        try:
            df = pd.read_csv(FILES[key])
            # Standardize column names
            df.columns = [c.strip() for c in df.columns]
            fsi_frames.append(df)
            print(f"  Loaded {key}: {len(df)} rows, year(s): {df['Year'].unique()}")
        except Exception as e:
            print(f"  Skipped {key}: {e}")
    
    if fsi_frames:
        fsi = pd.concat(fsi_frames, ignore_index=True)
        
        # Standardize column names
        fsi = fsi.rename(columns={
            'S1: Demographic Pressures': 'demographic_pressures',
            'S2: Refugees and IDPs': 'refugees_idps',
            'C3: Group Grievance': 'group_grievance',
            'E3: Human Flight and Brain Drain': 'human_flight',
            'E2: Economic Inequality': 'economic_inequality',
            'E1: Economy': 'economy',
            'P1: State Legitimacy': 'state_legitimacy',
            'P2: Public Services': 'public_services',
            'P3: Human Rights': 'human_rights',
            'C1: Security Apparatus': 'security_apparatus',
            'C2: Factionalized Elites': 'factionalized_elites',
            'X1: External Intervention': 'external_intervention',
            'Total': 'total_score',
            'Country': 'country_name',
            'Year': 'year'
        })
        
        # Map country codes
        code_map = dict(zip(countries_df['country_name'], countries_df['country_code']))
        fsi['country_code'] = fsi['country_name'].map(code_map)
        fsi_me = fsi.dropna(subset=['country_code'])
        
        # Select final columns
        fsi_cols = ['country_code', 'year', 'total_score', 'demographic_pressures',
                    'refugees_idps', 'group_grievance', 'human_flight', 
                    'economic_inequality', 'economy', 'state_legitimacy',
                    'public_services', 'human_rights', 'security_apparatus',
                    'factionalized_elites', 'external_intervention']
        fsi_me = fsi_me[[c for c in fsi_cols if c in fsi_me.columns]]
        
        con.execute("DROP TABLE IF EXISTS fragile_states_index")
        con.execute("""
            CREATE TABLE fragile_states_index AS SELECT * FROM fsi_me
        """)
        
        row_count = con.execute('SELECT COUNT(*) FROM fragile_states_index').fetchone()[0]
        print(f"FRAGILE_STATES_INDEX table created: {row_count:,} rows")
        logger.info(f"FRAGILE_STATES_INDEX table created: {row_count} rows")
        
except Exception as e:
    print(f"ERROR loading FSI: {e}")
    logger.error(f"FSI loading failed: {e}")

In [ ]:
try:
    fsi_frames = []
    fsi_keys = [k for k in FILES if k.startswith('fsi_')]
    
    for key in fsi_keys:
        try:
            df = pd.read_csv(FILES[key])
            # Standardize column names
            df.columns = [c.strip() for c in df.columns]
            fsi_frames.append(df)
            print(f"  Loaded {key}: {len(df)} rows, year(s): {df['Year'].unique()}")
        except Exception as e:
            print(f"  Skipped {key}: {e}")
    
    if fsi_frames:
        fsi = pd.concat(fsi_frames, ignore_index=True)
        
        # Standardize column names
        fsi = fsi.rename(columns={
            'S1: Demographic Pressures': 'demographic_pressures',
            'S2: Refugees and IDPs': 'refugees_idps',
            'C3: Group Grievance': 'group_grievance',
            'E3: Human Flight and Brain Drain': 'human_flight',
            'E2: Economic Inequality': 'economic_inequality',
            'E1: Economy': 'economy',
            'P1: State Legitimacy': 'state_legitimacy',
            'P2: Public Services': 'public_services',
            'P3: Human Rights': 'human_rights',
            'C1: Security Apparatus': 'security_apparatus',
            'C2: Factionalized Elites': 'factionalized_elites',
            'X1: External Intervention': 'external_intervention',
            'Total': 'total_score',
            'Country': 'country_name',
            'Year': 'year'
        })
        
        # Map country codes
        code_map = dict(zip(countries_df['country_name'], countries_df['country_code']))
        fsi['country_code'] = fsi['country_name'].map(code_map)
        fsi_me = fsi.dropna(subset=['country_code'])
        
        # Select final columns
        fsi_cols = ['country_code', 'year', 'total_score', 'demographic_pressures',
                    'refugees_idps', 'group_grievance', 'human_flight', 
                    'economic_inequality', 'economy', 'state_legitimacy',
                    'public_services', 'human_rights', 'security_apparatus',
                    'factionalized_elites', 'external_intervention']
        fsi_me = fsi_me[[c for c in fsi_cols if c in fsi_me.columns]]
        
        con.execute("DROP TABLE IF EXISTS fragile_states_index")
        con.execute("""
            CREATE TABLE fragile_states_index AS SELECT * FROM fsi_me
        """)
        
        row_count = con.execute('SELECT COUNT(*) FROM fragile_states_index').fetchone()[0]
        print(f"FRAGILE_STATES_INDEX table created: {row_count:,} rows")
        logger.info(f"FRAGILE_STATES_INDEX table created: {row_count} rows")
        
except Exception as e:
    print(f"ERROR loading FSI: {e}")
    logger.error(f"FSI loading failed: {e}")

## Summary

In [ ]:
print("\n" + "="*60)
print("DATABASE SUMMARY")
print("="*60)
# summaries

tables = con.execute("SHOW TABLES").fetchall()
for (table_name,) in tables:
    count = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    cols = con.execute(f"SELECT * FROM {table_name} LIMIT 0").description
    num_cols = len(cols)
    print(f"  {table_name}: {count:,} rows x {num_cols} columns")

print(f"\nDatabase saved to: {DB_PATH}")
logger.info("All tables loaded successfully")

## Sanity Check

In [ ]:
# sanity check
print("\n--- Sample Query: ACLED events by country ---")
result = con.execute("""
    SELECT c.country_name, COUNT(*) as event_count, SUM(a.fatalities) as total_fatalities
    FROM acled_events a
    JOIN countries c ON a.country_code = c.country_code
    GROUP BY c.country_name
    ORDER BY total_fatalities DESC
    LIMIT 10
""").fetchdf()
print(result.to_string(index=False))

print("\n--- Sample Query: Military spending for top conflict countries ---")
try:
    result2 = con.execute("""
        SELECT c.country_name, m.year, m.spending_current_usd
        FROM military_spending m
        JOIN countries c ON m.country_code = c.country_code
        WHERE m.year >= 2019
        ORDER BY m.spending_current_usd DESC
        LIMIT 10
    """).fetchdf()
    print(result2.to_string(index=False))
except Exception as e:
    print(f"Query error: {e}")

